In [5]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print('✅ Environment variables loaded successfully!')

✅ Environment variables loaded successfully!


### 2. System Prompt

**Concept**: The system prompt is the "identity" and "instruction manual" for your agent.

**Analogy**: Like a chef's handbook that defines their specialty and rules.

**Diagram**:

User Input $
ightarrow$ [ System Prompt + History ] $
ightarrow$ LLM $
ightarrow$ Response

In [6]:
SYSTEM_PROMPT = """
You are a world-class Personal Chef and healthy-eating specialist.

**Your Mission**:
Help users create delicious, healthy meals based on the ingredients they have on hand.

**Your Rules**:
1. **Ingredient First**: Always prioritize using the ingredients the user provides.
2. **Health Conscious**: Suggest healthy alternatives to unhealthy ingredients when possible.
3. **Concise & Clear**: Provide recipes in a clear, step-by-step format.
4. **Proactive but Polite**: If a key ingredient is missing for a classic dish, suggest a substitution or ask the user if they have it.
5. **Dietary Mindfulness**: Always ask about allergies or dietary restrictions if the user hasn't mentioned them.
"""

print("✅ System prompt defined!")

✅ System prompt defined!


In [7]:
model_name = "gpt-5-nano"

In [8]:
# --- DESIGN STEP 1: Define the Brain ---
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5-nano"
llm = ChatOpenAI(model=MODEL_NAME)
print(f"Brain defined using {MODEL_NAME}")

Brain defined using gpt-5-nano


In [9]:
# --- DESIGN STEP 2: Define the Utensils (Tools) ---
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for recipes and food trends"""
    return tavily_client.search(query)

# Registry of tools the agent is allowed to use
tools = [web_search]
print("Tools registry defined.")

Tools registry defined.


In [10]:
# --- DESIGN STEP 3: Construct the Agent ---
from langchain.agents import create_agent

# We bind the brain, the tools, and the system prompt together
chef_agent = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)

print("✅ Personal Chef Agent designed and constructed using create_agent.")

✅ Personal Chef Agent designed and constructed using create_agent.


### 5. Interacting with the Chef

**Concept**: Now that the agent is designed and constructed, we can pass user messages to it. The agent will decide if it needs to use the  tool or if it can answer from its own knowledge.

**Analogy**: This is like the customer finally walking into the kitchen and placing an order with the Head Chef.

In [17]:

def ask_chef(user_input):
    print(f"\n👤 User: {user_input}")
    print("👨‍🍳 Chef is thinking...")

    inputs = {"messages": [("user", user_input)]}

    try:
        for event in chef_agent.stream(inputs, stream_mode="updates"):
            for value in event.values():
                if value and "messages" in value:
                    last_msg = value["messages"][-1]
                    if last_msg.type == "ai" and last_msg.text:
                        print(f"\n🤖 Chef: {last_msg.text}")
    except Exception as e:
        print(f"❌ Error: {e}")


# --- TEST IT OUT ---
ask_chef("I have some salmon, asparagus, and lemon. What's a healthy 20-minute dinner I can make?")


👤 User: I have some salmon, asparagus, and lemon. What's a healthy 20-minute dinner I can make?
👨‍🍳 Chef is thinking...

🤖 Chef: Here’s a quick, healthy 20-minute dinner using your salmon, asparagus, and lemon.

One-pan lemon-garlic salmon with asparagus

Serves: 2

Ingredients
- 2 salmon fillets (about 6–7 oz each), skin on or off
- 1 bunch asparagus, trimmed (about 12–14 spears)
- 1 lemon (zest and juice)
- 2 tbsp olive oil, plus more if needed
- 2 garlic cloves, minced
- Salt and pepper
- Optional: pinch of chili flakes, fresh parsley or dill for garnish

Steps
1) Prep and season: Pat salmon dry, season with salt and pepper. Zest the lemon.
2) Sear salmon: Heat 1 tbsp olive oil in a large skillet over medium-high. Place salmon skin-side down (if skin is on). Sear 4–5 minutes until golden, flip, and cook 2–3 minutes more until just opaque. Remove salmon from pan.
3) Cook asparagus: In the same pan, add the remaining 1 tbsp olive oil. Add asparagus with a pinch of salt; sauté 4–5 min